# MyFirstNEURON — Colab Edition (Prototype 1)

This notebook is a Google-Colab-friendly, Python/Jupyter port of **MyFirstNEURON**, a NEURON demo
by Arthur Houweling and Terry Sejnowski (Salk Institute), based on experiments from
*Electrophysiology of the Neuron* by Huguenard & McCormick. Original files:
https://modeldb.science/3808

**Status:** first working increment, covering Experiment 5/6 ("impulse generation" — a
Hodgkin-Huxley action potential in a single-compartment cell). Once this is confirmed working,
the remaining experiments will be added the same way.

No local installation is needed — just run the cells top to bottom in Colab.

## 1. Setup (run once per Colab session)

Installs the `neuron` Python package, clones the original `.mod` mechanism files from the
[ModelDB GitHub mirror](https://github.com/ModelDBRepository/3808), and compiles them with
`nrnivmodl`.

In [4]:
%%capture
!pip install neuron

In [5]:
import os

MOD_SRC_DIR = "mfn_src"

if not os.path.isdir(MOD_SRC_DIR):
    !git clone --depth 1 https://github.com/ModelDBRepository/3808.git {MOD_SRC_DIR}

!cd {MOD_SRC_DIR} && nrnivmodl

'nrnivmodl' is not recognized as an internal or external command,
operable program or batch file.


In [3]:
from neuron import h
from neuron import load_mechanisms
import matplotlib.pyplot as plt

load_mechanisms(MOD_SRC_DIR)
h.load_file("stdrun.hoc")

print("NEURON is ready, mechanisms loaded from:", MOD_SRC_DIR)

ModuleNotFoundError: No module named 'neuron'

## 2. Build the cell

A single spherical compartment (same geometry as the original demo: total membrane area of
29000 &mu;m&sup2;), with passive leak channels (Na/K/Ca/Cl/Mg) and Hodgkin-Huxley
sodium/potassium channels.

In [ ]:
import math

soma = h.Section(name="soma")
soma.L = 290 / math.pi
soma.diam = 100
soma.nseg = 1

soma.insert("leak")
soma.insert("HH")

h.celsius = 35

# ionic concentrations (mM), as in the original e5.par
soma(0.5).nai = 31
soma(0.5).nao = 145
soma(0.5).ki = 135
soma(0.5).ko = 3.1

print("soma area (um^2):", h.area(0.5, sec=soma))

## 3. Stimulus and recording

In [ ]:
stim = h.IClamp(soma(0.5))

t_vec = h.Vector().record(h._ref_t)
v_vec = h.Vector().record(soma(0.5)._ref_v)

## 4. Run once (sanity check)

Parameters below match the original Experiment 5 (`e5.par`): a 50 ms, 2 nA current step
should evoke a train of action potentials.

In [ ]:
def run_experiment(gnabar_HH=0.069, gkbar_HH=0.0069, pna_leak=2.07e-7, pk_leak=3.45e-6,
                    stim_delay=10, stim_dur=50, stim_amp=2, tstop=80, v_init=-65, dt=0.05):
    soma(0.5).gnabar_HH = gnabar_HH
    soma(0.5).gkbar_HH = gkbar_HH
    soma(0.5).pna_leak = pna_leak
    soma(0.5).pk_leak = pk_leak

    stim.delay = stim_delay
    stim.dur = stim_dur
    stim.amp = stim_amp

    h.tstop = tstop
    h.v_init = v_init
    h.dt = dt
    h.run()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(t_vec, v_vec)
    ax.set_xlabel("time (ms)")
    ax.set_ylabel("membrane potential (mV)")
    ax.set_title("Experiment 5/6: impulse generation")
    ax.set_ylim(-100, 50)
    plt.show()

run_experiment()

## 5. Interactive exploration

Use the sliders to explore how the stimulus and channel densities affect firing —
similar to adjusting parameters in the original xpanel GUI.

In [ ]:
from ipywidgets import interact, FloatSlider

interact(
    run_experiment,
    gnabar_HH=FloatSlider(value=0.069, min=0, max=0.2, step=0.001, readout_format=".3f"),
    gkbar_HH=FloatSlider(value=0.0069, min=0, max=0.02, step=0.0005, readout_format=".4f"),
    pna_leak=FloatSlider(value=2.07e-7, min=0, max=1e-6, step=1e-8, readout_format=".2e"),
    pk_leak=FloatSlider(value=3.45e-6, min=0, max=1e-5, step=1e-7, readout_format=".2e"),
    stim_delay=FloatSlider(value=10, min=0, max=50, step=1),
    stim_dur=FloatSlider(value=50, min=1, max=100, step=1),
    stim_amp=FloatSlider(value=2, min=0, max=5, step=0.1),
    tstop=FloatSlider(value=80, min=20, max=200, step=5),
    v_init=FloatSlider(value=-65, min=-90, max=-40, step=1),
    dt=FloatSlider(value=0.05, min=0.01, max=0.25, step=0.01),
);